# Proyecto 2 – Análisis Exploratorio
## Reto 11 – Negocios: LMSYS Chatbot Arena Human Preference Predictions

**CC3084 Data Science | Universidad del Valle de Guatemala | Semestre II 2026**

| Integrante | Carné |
|---|---|
| Mia Alejandra Fuentes Mérida | 23775 |
| Roberto José Barreda Siekavizza | 23354 |
| Javier Eduardo España Pacheco | 23361 |
| Angel Esteban Esquit Hernández | 23221 |

---

## Parte 1 – Introducción, Contexto y Carga Inicial de Datos

---
## 1. Situación Problemática

El surgimiento de los **Modelos de Lenguaje Grande (LLMs)** ha transformado radicalmente el panorama de los negocios digitales. Empresas como OpenAI, Google, Meta, Mistral y decenas de laboratorios más compiten lanzando modelos que prometen ser más capaces, más útiles y más alineados con las necesidades humanas. Sin embargo, esta proliferación genera un problema concreto para quienes deben tomar decisiones de negocio: **¿cómo saber cuál modelo es genuinamente mejor para sus usuarios?**

Las métricas técnicas tradicionales (perplexity, BLEU, ROUGE) no capturan bien la calidad percibida por usuarios reales. Por eso, la organización **LMSYS** (Large Model Systems Organization, UC Berkeley) creó **Chatbot Arena**: una plataforma donde usuarios anónimos interactúan simultáneamente con dos modelos LLM sin saber cuál es cuál, y al final votan por la respuesta que prefieren. Este enfoque —conocido como *blind pairwise evaluation* o evaluación por pares ciega— es considerado el estándar de oro para medir la calidad real de un LLM.

La relevancia de negocio es directa:

- **Costo operativo:** Distintos modelos tienen precios muy distintos por token (GPT-4o, Claude 3.5, Gemini Ultra vs. modelos open-source). Una empresa que integra LLMs en sus productos quiere el modelo con mejor relación calidad-precio desde la perspectiva del usuario final.
- **Experiencia de usuario:** En aplicaciones como asistentes virtuales, generación de contenido, soporte al cliente o educación, la preferencia humana es la métrica que más impacta la retención.
- **Estrategia de selección de modelos:** Anticipar la preferencia humana permite hacer *routing* inteligente: enviar queries simples a modelos baratos y queries complejos a modelos premium, optimizando costo y calidad.

Comprender **qué factores determinan que un usuario prefiera una respuesta sobre otra** es, por tanto, una pregunta de negocio con impacto económico real.

---
## 2. Problema Científico

> **¿Es posible identificar, a partir de características observables del prompt y de las respuestas generadas por dos LLMs, los patrones que predicen la preferencia humana en una evaluación ciega por pares?**

Esta pregunta engloba preguntas específicas exploradas durante el análisis:

- ¿La longitud de la respuesta influye en la preferencia del usuario?
- ¿Existen modelos sistemáticamente preferidos independientemente del oponente?
- ¿La posición del modelo (A o B) introduce sesgo en la evaluación?
- ¿Qué tipos de prompt generan más empates?
- ¿Existen diferencias estadísticamente significativas en el estilo de respuesta de los modelos ganadores?

---
## 3. Objetivos

### Objetivo General
Realizar un análisis exploratorio exhaustivo del dataset de Chatbot Arena para identificar patrones, variables relevantes y hallazgos que permitan comprender los factores que determinan la preferencia humana entre respuestas de modelos de lenguaje grande, sentando las bases para el desarrollo de un modelo predictivo.

### Objetivos Específicos

1. **Describir y caracterizar el dataset**: Identificar la estructura, tipos de variables, dimensiones, calidad de los datos (valores nulos, duplicados) y realizar las transformaciones de limpieza necesarias para el análisis.

2. **Analizar la distribución de variables cuantitativas y categóricas**: Estudiar la distribución de las longitudes de prompts y respuestas, la frecuencia de participación de cada modelo y la distribución del resultado (victoria de A, victoria de B, empate) mediante estadística descriptiva y visualizaciones.

3. **Identificar relaciones entre variables y el resultado**: Explorar correlaciones entre características de los textos (longitud, número de turnos) y el resultado de la batalla, detectar sesgos de posición y analizar qué modelos muestran mayor tasa de victoria, con el fin de generar hipótesis accionables para el modelado predictivo.

---
## 4. Descripción del Dataset

El dataset proviene de la competencia [LMSYS - Chatbot Arena Human Preference Predictions](https://www.kaggle.com/competitions/lmsys-chatbot-arena) en Kaggle.

Cada fila representa una **batalla** entre dos LLMs en la que un usuario humano evaluó ambas respuestas y emitió su preferencia. Los modelos compiten de forma anónima (el usuario no sabe cuál es cuál al momento de votar).

### Archivos del dataset

| Archivo | Descripción |
|---|---|
| `train.csv` | Batallas con el resultado (winner) etiquetado — datos de entrenamiento |
| `test.csv` | Batallas sin resultado — para predicción en la competencia |
| `sample_submission.csv` | Formato de entrega para Kaggle |

### Columnas de `train.csv`

| Columna | Tipo | Descripción |
|---|---|---|
| `id` | string | Identificador único de la batalla |
| `model_a` | string | Nombre del primer LLM en la batalla |
| `model_b` | string | Nombre del segundo LLM en la batalla |
| `prompt` | string (JSON list) | Turnos del prompt del usuario (puede ser multi-turno) |
| `response_a` | string (JSON list) | Respuestas generadas por `model_a` |
| `response_b` | string (JSON list) | Respuestas generadas por `model_b` |
| `winner_model_a` | int (0/1) | 1 si el usuario prefirió la respuesta de `model_a` |
| `winner_model_b` | int (0/1) | 1 si el usuario prefirió la respuesta de `model_b` |
| `winner_tie` | int (0/1) | 1 si el usuario declaró empate |

> **Nota:** Las tres columnas `winner_*` son mutuamente excluyentes. Exactamente una vale 1 en cada fila.

---
## 5. Configuración del Ambiente

In [1]:
# Instalación de dependencias (ejecutar solo si es necesario)
# !pip install -r ../requirements.txt

In [2]:
import sys
from pathlib import Path

# Agregar el directorio raíz al path para importar src/
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src import config as cfg
from src import load

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print(' Librerías cargadas correctamente')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   ROOT    {ROOT}')

 Librerías cargadas correctamente
   pandas  2.1.4
   numpy   1.26.4
   ROOT    /home/javier-espana/Escritorio/CC3084-PRY2


---
## 6. Descarga del Dataset

El dataset se descarga desde Kaggle. Para ello necesitás tener tu API key configurada.

In [3]:
# Opción A: Descargar con kagglehub (recomendado)
# Requiere: pip install kagglehub
# Y tener ~/.kaggle/kaggle.json con tu API key de Kaggle

import kagglehub
import shutil

RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Descargando dataset de lmsys-chatbot-arena...')
download_path = Path(kagglehub.competition_download('lmsys-chatbot-arena'))
print(f'Dataset descargado en: {download_path}')

# Copiar archivos a data/raw/ si no están ya ahí
for f in download_path.glob('*.csv'):
    dest = RAW_DIR / f.name
    if not dest.exists():
        shutil.copy2(f, dest)
        print(f'  Copiado: {f.name}')
    else:
        print(f'  Ya existe: {f.name}')

print('\n Archivos disponibles en data/raw/:')
for f in sorted(RAW_DIR.glob('*.csv')):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name:30s}  {size_mb:7.1f} MB')

Descargando dataset de lmsys-chatbot-arena...


/home/javier-espana/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UnauthenticatedError: User is not authenticated

---
## 7. Carga del Dataset

In [ ]:
# Cargamos train.csv completo
df = load.load_train()
print(f'Shape de train.csv: {df.shape[0]:,} filas × {df.shape[1]} columnas')

In [ ]:
# Primeras filas
df.head(3)

---
## 8. Primera Exploración del Dataset

In [ ]:
# Tipos de datos y valores no nulos
df.info()

In [ ]:
# Estadísticas de las columnas numéricas (columnas winner_*)
df[cfg.WINNER_COLS].describe()

In [ ]:
# Conteo de valores nulos por columna
nulos = df.isnull().sum().rename('nulos')
nulos_pct = (df.isnull().mean() * 100).round(2).rename('% nulos')
pd.concat([nulos, nulos_pct], axis=1)

In [ ]:
# Distribución de ganadores (rápido vistazo)
ganadores = pd.Series({
    'model_a gana': df[cfg.COL_WIN_A].sum(),
    'model_b gana': df[cfg.COL_WIN_B].sum(),
    'empate':       df[cfg.COL_WIN_TIE].sum(),
}, name='conteo')

ganadores_pct = (ganadores / len(df) * 100).round(2).rename('% del total')
resumen_ganadores = pd.concat([ganadores, ganadores_pct], axis=1)
print(resumen_ganadores.to_string())

In [ ]:
# Visualización rápida de la distribución de resultados
fig, ax = plt.subplots(figsize=(6, 4))
colores = ['#4C72B0', '#DD8452', '#8172B2']
bars = ax.bar(resumen_ganadores.index, resumen_ganadores['conteo'], color=colores, edgecolor='white')

for bar, pct in zip(bars, ganadores_pct):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{pct:.1f}%',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_title('Distribución de resultados en Chatbot Arena (train)', fontsize=13, pad=12)
ax.set_ylabel('Número de batallas')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, ganadores.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.savefig('../docs/fig_distribucion_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Figura guardada en docs/fig_distribucion_resultado.png')

In [ ]:
# Modelos únicos en el dataset
modelos_a = set(df[cfg.COL_MODEL_A].dropna().unique())
modelos_b = set(df[cfg.COL_MODEL_B].dropna().unique())
todos_modelos = modelos_a | modelos_b

print(f'Modelos únicos como model_a: {len(modelos_a)}')
print(f'Modelos únicos como model_b: {len(modelos_b)}')
print(f'Total de modelos distintos en el dataset: {len(todos_modelos)}')
print('\nListado de modelos:')
for m in sorted(todos_modelos):
    print(f'  {m}')

---
## 9. Resumen de Hallazgos de la Carga Inicial

En esta primera exploración encontramos:

- El dataset tiene **`N` filas y 9 columnas**, representando cada fila una batalla entre dos LLMs.
- Las columnas de texto (`prompt`, `response_a`, `response_b`) están almacenadas como **JSON strings** que contienen listas de turnos, lo que requerirá un proceso de extracción de texto en la Parte 2.
- Se detectaron **valores nulos** en las columnas de respuesta (`response_a`, `response_b`), que deberán ser tratados.
- La distribución de resultados muestra que **model_a y model_b ganan proporciones similares**, con una tasa de empate notable.
- Existen múltiples modelos distintos (incluyendo GPT-4, Claude, Gemini, Llama, Mixtral, entre otros).

**Próximos pasos (Parte 2):** Limpieza de datos, extracción de texto de las columnas JSON, derivación de la columna `winner` unificada y cálculo de features de longitud.